In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1FGvABsBFk-5eeJCzd71PvXfHOcJPtgO2", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))


# Notebook 3: Data Parallelism

**Vizuara AI Pods | GPU Programming Course | Pod 3: Recomputation, Accumulation, and Data Parallelism**

---

In this notebook, we will build a deep understanding of **data parallelism** -- the most widely used multi-GPU training strategy, where we replicate the model on every GPU, split the data, and synchronize gradients via AllReduce.

Since Colab provides only a single GPU, we will **simulate** multi-GPU data parallelism on a single GPU to understand the mechanics. This gives us full control over every step.

By the end of this notebook, you will:

1. Understand the complete data parallelism workflow (replicate, shard, compute, sync)
2. Implement a manual AllReduce simulation to see gradient synchronization in action
3. Verify that synchronized gradients produce identical weight updates
4. Understand the communication overhead and scaling efficiency
5. Combine data parallelism with checkpointing and accumulation

**Prerequisites:** Notebooks 1 and 2 (Checkpointing, Gradient Accumulation).

**Estimated time:** 40-50 minutes

**Runtime:** Make sure you are using a GPU runtime in Colab: Runtime > Change runtime type > T4 GPU

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Setup

In [ ]:
!pip install -q torch matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import time
import copy
import gc

assert torch.cuda.is_available(), "No GPU found! Please enable GPU in Runtime > Change runtime type."
device = torch.device('cuda')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"\nNote: We will SIMULATE multi-GPU data parallelism on this single GPU.")

In [ ]:
#@title 🎧 Transition: Part1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_part1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 1: The Data Parallelism Workflow

Data parallelism has five steps:

1. **Replicate** the model to N GPUs (all copies start identical)
2. **Shard** the mini-batch: each GPU gets `batch_size / N` samples
3. **Forward + Backward** independently on each GPU
4. **AllReduce** the gradients (average across all GPUs)
5. **Update** weights identically on each GPU

Let's simulate this step by step with N=4 "virtual GPUs".

In [ ]:
#@title 🎧 Code Walkthrough: Simple Model
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_simple_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class SimpleModel(nn.Module):
    """A simple model for data parallelism experiments."""
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)

    def forward(self, x):
        x = F.gelu(self.ln1(self.fc1(x)))
        x = F.gelu(self.ln2(self.fc2(x)))
        return self.fc3(x)


# Create the "world" -- N virtual GPU replicas
NUM_GPUS = 4
MICRO_BATCH_PER_GPU = 8
TOTAL_BATCH = NUM_GPUS * MICRO_BATCH_PER_GPU  # 32

print(f"Simulating {NUM_GPUS} GPUs")
print(f"Micro-batch per GPU: {MICRO_BATCH_PER_GPU}")
print(f"Effective batch size: {TOTAL_BATCH}")

In [ ]:
#@title 🎧 Code Walkthrough: Replicate Models
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_replicate_models.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 1: Create the master model and replicate it to N "GPUs"
torch.manual_seed(42)
master_model = SimpleModel().to(device)

# Create N identical copies (simulating model replication)
gpu_models = [copy.deepcopy(master_model) for _ in range(NUM_GPUS)]

# Verify all replicas are identical
for i in range(1, NUM_GPUS):
    for (name_0, p_0), (name_i, p_i) in zip(
        gpu_models[0].named_parameters(), gpu_models[i].named_parameters()
    ):
        assert torch.equal(p_0, p_i), f"Mismatch in {name_0} between GPU 0 and GPU {i}"

print(f"Step 1: Created {NUM_GPUS} identical model replicas.")
print(f"  Parameters per replica: {sum(p.numel() for p in master_model.parameters()):,}")
print(f"  Total parameters across all GPUs: {sum(p.numel() for p in master_model.parameters()) * NUM_GPUS:,}")
print(f"  (This is the memory cost of data parallelism -- N full copies!)")

In [ ]:
#@title 🎧 Code Walkthrough: Shard Data
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_shard_data.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 2: Create a full batch and shard it across GPUs
torch.manual_seed(123)
full_batch_x = torch.randn(TOTAL_BATCH, 64, device=device)
full_batch_y = torch.randint(0, 10, (TOTAL_BATCH,), device=device)

# Shard the data -- each GPU gets a different slice
data_shards = []
label_shards = []
for i in range(NUM_GPUS):
    start = i * MICRO_BATCH_PER_GPU
    end = start + MICRO_BATCH_PER_GPU
    data_shards.append(full_batch_x[start:end])
    label_shards.append(full_batch_y[start:end])

print(f"Step 2: Sharded {TOTAL_BATCH} samples across {NUM_GPUS} GPUs.")
for i in range(NUM_GPUS):
    print(f"  GPU {i}: samples [{i*MICRO_BATCH_PER_GPU}:{(i+1)*MICRO_BATCH_PER_GPU}] "
          f"(shape: {data_shards[i].shape})")

In [ ]:
#@title 🎧 Code Walkthrough: Forward Backward
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_forward_backward.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 3: Each GPU computes forward + backward independently
local_gradients = []  # Will hold the gradient dict for each GPU

for i in range(NUM_GPUS):
    model = gpu_models[i]
    model.zero_grad()

    # Forward pass on this GPU's shard
    outputs = model(data_shards[i])
    loss = F.cross_entropy(outputs, label_shards[i])
    loss.backward()

    # Collect local gradients
    grads = {name: p.grad.clone() for name, p in model.named_parameters()}
    local_gradients.append(grads)

    print(f"  GPU {i}: loss = {loss.item():.4f}")

print(f"\nStep 3: Each GPU computed gradients on its local data shard.")
print(f"  Notice the losses differ -- each GPU sees different data!")

In [ ]:
#@title 🎧 Code Walkthrough: Allreduce Gradients
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_allreduce_gradients.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 4: AllReduce -- average gradients across all GPUs

def manual_allreduce(local_gradients, num_gpus):
    """
    Simulate AllReduce: average gradients across all GPUs.

    In real distributed training, this happens over the network (NCCL).
    Here we do it explicitly to understand the mechanics.
    """
    param_names = list(local_gradients[0].keys())
    averaged_gradients = {}

    for name in param_names:
        # Sum gradients from all GPUs
        grad_sum = sum(local_gradients[i][name] for i in range(num_gpus))
        # Average
        averaged_gradients[name] = grad_sum / num_gpus

    return averaged_gradients


# Run AllReduce
avg_gradients = manual_allreduce(local_gradients, NUM_GPUS)

print("Step 4: AllReduce complete.")
print("\nGradient comparison (showing fc1.weight gradient norms):")
print("-" * 50)
for i in range(NUM_GPUS):
    local_norm = local_gradients[i]['fc1.weight'].norm().item()
    print(f"  GPU {i} local gradient norm:  {local_norm:.6f}")
avg_norm = avg_gradients['fc1.weight'].norm().item()
print(f"  Averaged gradient norm:      {avg_norm:.6f}")
print("\nAfter AllReduce, every GPU gets the SAME averaged gradient.")

In [ ]:
#@title 🎧 Code Walkthrough: Update Weights
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_update_weights.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Step 5: Apply the same averaged gradient to all model replicas

lr = 0.01

for i in range(NUM_GPUS):
    model = gpu_models[i]
    # Replace local gradients with averaged gradients
    for name, p in model.named_parameters():
        p.grad = avg_gradients[name].clone()

# Create optimizers and step (all apply the same update)
optimizers = [torch.optim.SGD(m.parameters(), lr=lr) for m in gpu_models]
for opt in optimizers:
    opt.step()

# Verify all models are still identical after the update
all_synced = True
for i in range(1, NUM_GPUS):
    for (name_0, p_0), (name_i, p_i) in zip(
        gpu_models[0].named_parameters(), gpu_models[i].named_parameters()
    ):
        if not torch.equal(p_0, p_i):
            all_synced = False
            print(f"  DESYNC in {name_0} between GPU 0 and GPU {i}!")

print(f"Step 5: Weight update complete.")
print(f"  All model replicas synchronized: {all_synced}")
print(f"\nThis is the key invariant of data parallelism:")
print(f"  Same averaged gradient + same optimizer = identical weights on all GPUs.")

In [ ]:
#@title 🎧 Transition: Part2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_part2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 2: Comparing Data Parallel vs Single-GPU Training

Let's verify that data parallel training produces the exact same result as single-GPU training with the full batch.

In [ ]:
#@title 🎧 Code Walkthrough: Comparison Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_comparison_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Reset everything for a clean comparison
torch.manual_seed(42)

# METHOD 1: Single-GPU full batch training
model_single = SimpleModel().to(device)
opt_single = torch.optim.SGD(model_single.parameters(), lr=0.01)

# Create data
torch.manual_seed(123)
full_x = torch.randn(32, 64, device=device)
full_y = torch.randint(0, 10, (32,), device=device)

# Single GPU training step
opt_single.zero_grad()
out = model_single(full_x)
loss_single = F.cross_entropy(out, full_y)
loss_single.backward()
opt_single.step()

print(f"Single GPU loss: {loss_single.item():.6f}")

# METHOD 2: Simulated 4-GPU data parallel training
torch.manual_seed(42)
gpu_models_2 = [SimpleModel().to(device) for _ in range(4)]

local_grads_2 = []
losses_dp = []

for i in range(4):
    model = gpu_models_2[i]
    model.zero_grad()
    shard_x = full_x[i*8:(i+1)*8]
    shard_y = full_y[i*8:(i+1)*8]
    out = model(shard_x)
    loss = F.cross_entropy(out, shard_y)
    loss.backward()
    local_grads_2.append({n: p.grad.clone() for n, p in model.named_parameters()})
    losses_dp.append(loss.item())

# AllReduce
avg_grads_2 = manual_allreduce(local_grads_2, 4)
for model in gpu_models_2:
    for name, p in model.named_parameters():
        p.grad = avg_grads_2[name].clone()

opts_dp = [torch.optim.SGD(m.parameters(), lr=0.01) for m in gpu_models_2]
for opt in opts_dp:
    opt.step()

avg_loss_dp = sum(losses_dp) / 4
print(f"Data parallel avg loss: {avg_loss_dp:.6f}")

# Compare weights after one step
print(f"\n{'Parameter':<25} {'Max Weight Difference':>25}")
print("-" * 55)
all_match = True
for (name_s, p_s), (name_d, p_d) in zip(
    model_single.named_parameters(), gpu_models_2[0].named_parameters()
):
    diff = (p_s - p_d).abs().max().item()
    match = diff < 1e-5
    all_match = all_match and match
    print(f"{name_s:<25} {diff:>25.2e}  {'OK' if match else 'MISMATCH'}")

print(f"\nWeights match exactly: {all_match}")
print("Data parallel training is mathematically equivalent to single-GPU training!")

del model_single, gpu_models_2
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#@title 🎧 Transition: Part3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_part3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 3: Communication Overhead Analysis

In real data parallelism, the AllReduce operation is the main bottleneck. Each GPU must send and receive gradient tensors over the network. Let's analyze the communication cost.

In [ ]:
#@title 🎧 Code Walkthrough: Comm Cost Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_comm_cost_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_communication_cost(model, num_gpus_list):
    """
    Analyze the communication cost of AllReduce for a given model.
    """
    total_params = sum(p.numel() for p in model.parameters())
    # In mixed precision (FP16), each parameter = 2 bytes
    # AllReduce with ring algorithm: each GPU sends and receives ~2 * P * (N-1)/N bytes

    print(f"Model parameters: {total_params:,}")
    print(f"Gradient size (FP16): {total_params * 2 / 1024**2:.1f} MB")
    print(f"Gradient size (FP32): {total_params * 4 / 1024**2:.1f} MB")
    print()

    results = []
    print(f"{'GPUs':>6} {'Data/GPU (MB)':>14} {'Ring AllReduce (MB)':>20} {'Time (ms) @ 25GB/s':>20}")
    print("-" * 65)

    for N in num_gpus_list:
        # Ring AllReduce: each GPU sends (N-1)/N * size in reduce-scatter
        # then (N-1)/N * size in all-gather = 2 * (N-1)/N * size total
        grad_bytes = total_params * 2  # FP16
        ring_bytes = 2 * grad_bytes * (N - 1) / N
        ring_mb = ring_bytes / 1024**2

        # Assume 25 GB/s inter-GPU bandwidth (NVLink-like)
        time_ms = ring_bytes / (25 * 1024**3) * 1000

        results.append((N, grad_bytes / 1024**2, ring_mb, time_ms))
        print(f"{N:>6} {grad_bytes/1024**2:>14.1f} {ring_mb:>20.1f} {time_ms:>20.2f}")

    return results

# Analyze for our small model
print("=== Small model (SimpleModel) ===")
small_model = SimpleModel().to(device)
results_small = analyze_communication_cost(small_model, [2, 4, 8, 16, 32])

print("\n=== Larger model (e.g., 1.3B parameters) ===")
# Simulate a larger model
class LargeModelProxy:
    def parameters(self):
        # 1.3B parameters
        return [torch.empty(1_300_000_000)]

results_large = analyze_communication_cost(LargeModelProxy(), [2, 4, 8, 16, 32, 64])

del small_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#@title 🎧 What to Look For: Scaling Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_scaling_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize scaling efficiency
num_gpus = [1, 2, 4, 8, 16, 32]

# Simulate scaling: assume 50ms compute per GPU, communication cost from above
compute_time_ms = 50  # base compute time per step

# For a 1.3B model at 25 GB/s bandwidth
grad_size_bytes = 1.3e9 * 2  # FP16

ideal_throughput = []
actual_throughput = []
efficiencies = []

for N in num_gpus:
    ideal = N  # ideal linear scaling
    ideal_throughput.append(ideal)

    if N == 1:
        comm_time = 0
    else:
        ring_bytes = 2 * grad_size_bytes * (N - 1) / N
        comm_time = ring_bytes / (25 * 1024**3) * 1000  # ms

    total_time = compute_time_ms + comm_time
    actual = N * compute_time_ms / total_time  # effective speedup
    actual_throughput.append(actual)
    efficiencies.append(actual / ideal * 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Speedup
ax = axes[0]
ax.plot(num_gpus, ideal_throughput, 'k--', linewidth=1.5, label='Ideal (linear)')
ax.plot(num_gpus, actual_throughput, 'o-', color='#1E88E5', linewidth=2, markersize=8, label='Actual (with comm.)')
ax.fill_between(num_gpus, actual_throughput, ideal_throughput, alpha=0.15, color='#E53935')
ax.set_xlabel('Number of GPUs', fontsize=12)
ax.set_ylabel('Speedup (x)', fontsize=12)
ax.set_title('Data Parallelism Scaling (1.3B model)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Efficiency
ax = axes[1]
bars = ax.bar([str(n) for n in num_gpus], efficiencies, color='#43A047', alpha=0.8, width=0.5)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Number of GPUs', fontsize=12)
ax.set_ylabel('Scaling Efficiency (%)', fontsize=12)
ax.set_title('Scaling Efficiency', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.grid(True, alpha=0.3, axis='y')

for bar, eff in zip(bars, efficiencies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{eff:.0f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nThe gap between ideal and actual grows with more GPUs.")
print("This is the communication overhead from AllReduce.")
print("Techniques like gradient compression and overlap can reduce this gap.")

In [ ]:
#@title 🎧 Transition: Part4 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_part4_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 4: Full Data Parallel Training Loop

Let's put it all together in a complete simulated data parallel training loop and train a model to convergence.

In [ ]:
#@title 🎧 Code Walkthrough: Simulated Dp Train
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_simulated_dp_train.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def simulated_data_parallel_train(
    model_class, model_kwargs, dataset, num_gpus, micro_batch_per_gpu,
    num_epochs=10, lr=1e-3
):
    """
    Simulate data parallel training on a single GPU.

    This mimics exactly what PyTorch DistributedDataParallel does:
    1. Replicate model to N GPUs
    2. Shard data across GPUs (DistributedSampler)
    3. Forward+backward independently
    4. AllReduce gradients
    5. Identical optimizer step
    """
    effective_batch = num_gpus * micro_batch_per_gpu

    # Step 1: Create N model replicas
    torch.manual_seed(42)
    models = [model_class(**model_kwargs).to(device) for _ in range(num_gpus)]

    # All replicas must start identical
    state = models[0].state_dict()
    for m in models[1:]:
        m.load_state_dict(copy.deepcopy(state))

    optimizers = [torch.optim.AdamW(m.parameters(), lr=lr) for m in models]

    # DataLoader with effective batch size
    dataloader = DataLoader(dataset, batch_size=effective_batch, shuffle=True, drop_last=True)

    history = {'loss': [], 'accuracy': []}

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        epoch_correct = 0
        epoch_total = 0
        num_steps = 0

        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            # Step 2: Shard data
            local_grads = []
            total_loss = 0.0
            total_correct = 0

            for i in range(num_gpus):
                start = i * micro_batch_per_gpu
                end = start + micro_batch_per_gpu
                x_shard = batch_x[start:end]
                y_shard = batch_y[start:end]

                # Step 3: Independent forward + backward
                models[i].zero_grad()
                out = models[i](x_shard)
                loss = F.cross_entropy(out, y_shard)
                loss.backward()

                total_loss += loss.item()
                total_correct += (out.argmax(1) == y_shard).sum().item()

                local_grads.append(
                    {n: p.grad.clone() for n, p in models[i].named_parameters()}
                )

            # Step 4: AllReduce
            avg_grads = manual_allreduce(local_grads, num_gpus)

            # Step 5: Apply averaged gradients and update
            for i in range(num_gpus):
                for name, p in models[i].named_parameters():
                    p.grad = avg_grads[name].clone()
                optimizers[i].step()

            epoch_loss += total_loss / num_gpus
            epoch_correct += total_correct
            epoch_total += batch_x.size(0)
            num_steps += 1

        avg_loss = epoch_loss / num_steps
        accuracy = epoch_correct / epoch_total * 100
        history['loss'].append(avg_loss)
        history['accuracy'].append(accuracy)

        # Verify sync invariant holds
        synced = all(
            torch.equal(p0, pi)
            for (_, p0), (_, pi) in zip(
                models[0].named_parameters(), models[-1].named_parameters()
            )
        )

        print(f"  Epoch {epoch+1:>2}/{num_epochs} | Loss: {avg_loss:.4f} | "
              f"Acc: {accuracy:.1f}% | Synced: {synced}")

    return history, models[0]

print("Simulated data parallel training function ready.")

In [ ]:
#@title 🎧 Code Walkthrough: Run Training
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_run_training.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Create dataset
torch.manual_seed(42)
num_samples = 4096
input_dim = 64
num_classes = 10

centers = torch.randn(num_classes, input_dim) * 3
labels = torch.randint(0, num_classes, (num_samples,))
data = centers[labels] + torch.randn(num_samples, input_dim) * 0.5
dataset = TensorDataset(data, labels)

# Train with different numbers of simulated GPUs
gpu_configs = [
    {"num_gpus": 1, "micro_bs": 32, "label": "1 GPU (BS=32)"},
    {"num_gpus": 2, "micro_bs": 16, "label": "2 GPUs (BS=16 each)"},
    {"num_gpus": 4, "micro_bs": 8,  "label": "4 GPUs (BS=8 each)"},
]

all_results = []

for cfg in gpu_configs:
    print(f"\nTraining with {cfg['label']}:")
    print("-" * 60)
    history, _ = simulated_data_parallel_train(
        SimpleModel, {'input_dim': 64, 'hidden_dim': 128, 'output_dim': 10},
        dataset, num_gpus=cfg['num_gpus'], micro_batch_per_gpu=cfg['micro_bs'],
        num_epochs=8, lr=1e-3
    )
    all_results.append((cfg['label'], history))

In [ ]:
#@title 🎧 What to Look For: Training Plots
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_training_plots.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#E53935', '#1E88E5', '#43A047']

ax = axes[0]
for (label, hist), color in zip(all_results, colors):
    ax.plot(hist['loss'], 'o-', color=color, linewidth=2, markersize=6, label=label)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Training Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

ax = axes[1]
for (label, hist), color in zip(all_results, colors):
    ax.plot(hist['accuracy'], 'o-', color=color, linewidth=2, markersize=6, label=label)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Training Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nAll configurations converge similarly because they have the same effective batch size.")
print("The difference in practice would be wall-clock time: more GPUs = faster per epoch.")

In [ ]:
#@title 🎧 Transition: Part5 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_part5_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 5: PyTorch DistributedDataParallel (DDP) Reference

In real multi-GPU training, you use `torch.nn.parallel.DistributedDataParallel` (DDP). Here is the canonical pattern. While we cannot run this on Colab (single GPU), understanding the code is essential.

In [ ]:
#@title 🎧 Code Walkthrough: Ddp Template
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_ddp_template.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# This is REFERENCE CODE -- it requires multiple GPUs to run.
# Study the pattern; do NOT try to execute it on Colab.

DDP_TRAINING_TEMPLATE = '''
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler

def train_ddp(rank, world_size):
    # Initialize process group
    dist.init_process_group(
        backend="nccl",       # NCCL is optimal for GPU communication
        rank=rank,             # This process's ID (0 to world_size-1)
        world_size=world_size  # Total number of processes/GPUs
    )
    torch.cuda.set_device(rank)

    # Create model and wrap with DDP
    model = MyModel().cuda(rank)
    model = DDP(model, device_ids=[rank])
    # DDP automatically handles:
    #   - Broadcasting initial weights from rank 0 to all ranks
    #   - AllReduce of gradients during backward()
    #   - Gradient bucketing for communication efficiency

    # DistributedSampler ensures each GPU gets different data
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    dataloader = DataLoader(dataset, batch_size=micro_batch_size, sampler=sampler)

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    for epoch in range(num_epochs):
        sampler.set_epoch(epoch)  # Ensure different shuffling each epoch

        for inputs, labels in dataloader:
            inputs = inputs.cuda(rank)
            labels = labels.cuda(rank)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()   # DDP auto-syncs gradients here!
            optimizer.step()

    dist.destroy_process_group()

# Launch with torchrun:
# torchrun --nproc_per_node=4 train.py
'''

print("DDP Training Template:")
print("=" * 60)
print(DDP_TRAINING_TEMPLATE)
print("Key points:")
print("1. DDP wraps the model and handles AllReduce automatically")
print("2. DistributedSampler shards data across GPUs")
print("3. The training loop looks almost identical to single-GPU code!")
print("4. Launch with 'torchrun' (handles process spawning)")

In [ ]:
#@title 🎧 Transition: Part6 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_part6_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Part 6: Putting It All Together -- The Complete Memory Toolkit

Let's visualize how all three techniques from this pod compose together.

In [ ]:
#@title 🎧 Code Walkthrough: Memory Toolkit Calc
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_memory_toolkit_calc.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Calculate the full example from the article:
# 1.3B transformer, 4x A100-40GB, target batch=256

print("COMPLETE EXAMPLE: Training a 1.3B Transformer")
print("=" * 65)
print()

# Model memory
params_b = 1.3  # billion
weights_gb = params_b * 2  # FP16
gradients_gb = params_b * 2  # FP16
optimizer_gb = params_b * 8  # Adam: 2 states * 4 bytes each
model_total_gb = weights_gb + gradients_gb + optimizer_gb

print(f"Step 0: Model memory budget")
print(f"  Weights (FP16):      {weights_gb:.1f} GB")
print(f"  Gradients (FP16):    {gradients_gb:.1f} GB")
print(f"  Optimizer (Adam):    {optimizer_gb:.1f} GB")
print(f"  Total model memory:  {model_total_gb:.1f} GB")
print(f"  Available for activations: {40 - model_total_gb:.1f} GB per A100")
print()

# Without optimization
print(f"Step 1: WITHOUT any optimization")
activation_per_sample = 0.75  # ~GB per sample for 1.3B model, seq=2048
bs_no_opt = int((40 - model_total_gb) / activation_per_sample)
print(f"  Max batch per GPU: ~{bs_no_opt} samples")
print(f"  Effective batch (4 GPUs): ~{bs_no_opt * 4} samples")
print(f"  Target batch: 256 -- INSUFFICIENT!")
print()

# With checkpointing
checkpoint_factor = 3  # ~3x reduction
activation_ckpt = activation_per_sample / checkpoint_factor
bs_ckpt = int((40 - model_total_gb) / activation_ckpt)
print(f"Step 2: With ACTIVATION CHECKPOINTING")
print(f"  Activation memory per sample: {activation_ckpt:.2f} GB (3x reduction)")
print(f"  Max micro-batch per GPU: ~{bs_ckpt} samples")
print()

# With data parallelism
num_gpus = 4
micro_batch = 4
dp_batch = num_gpus * micro_batch
print(f"Step 3: With DATA PARALLELISM ({num_gpus} GPUs)")
print(f"  Micro-batch per GPU: {micro_batch}")
print(f"  Batch per step: {dp_batch} samples")
print()

# With gradient accumulation
target_batch = 256
accum_steps = target_batch // dp_batch
print(f"Step 4: With GRADIENT ACCUMULATION")
print(f"  Target effective batch: {target_batch}")
print(f"  Accumulation steps: K = {target_batch} / {dp_batch} = {accum_steps}")
print(f"  Effective batch size: {num_gpus} * {micro_batch} * {accum_steps} = {num_gpus * micro_batch * accum_steps}")
print()
print("Result: All three techniques together achieve the target batch of 256!")

In [ ]:
#@title 🎧 What to Look For: Memory Toolkit Viz
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_memory_toolkit_viz.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the complete picture
fig, ax = plt.subplots(figsize=(10, 6))

techniques = ['No\nOptimization', 'Activation\nCheckpointing', '+ Data\nParallelism\n(4 GPUs)', '+ Gradient\nAccumulation\n(K=16)']
effective_bs = [bs_no_opt * 4, bs_ckpt * 4, dp_batch, target_batch]
colors_bar = ['#E53935', '#FF9800', '#1E88E5', '#43A047']

bars = ax.bar(techniques, effective_bs, color=colors_bar, alpha=0.85, width=0.55)
ax.axhline(y=target_batch, color='red', linestyle='--', linewidth=2, label=f'Target: {target_batch}')

for bar, bs in zip(bars, effective_bs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{bs}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Effective Batch Size', fontsize=12)
ax.set_title('Building Up to the Target Batch Size', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nEach technique contributes to reaching the target:")
print(f"  Checkpointing: allows micro-batch of {micro_batch} (vs ~{bs_no_opt} without)")
print(f"  Data parallelism: multiplies by {num_gpus}x (batch = {dp_batch})")
print(f"  Accumulation: multiplies by {accum_steps}x (batch = {target_batch})")

In [ ]:
#@title 🎧 Narration: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_todo_intro_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Exercises

### TODO Exercise 1: Gradient Desynchronization Detector

In real training, bugs can cause model replicas to drift apart (e.g., forgetting AllReduce, different random seeds). Build a function that checks whether model replicas are synchronized after each step.

In [ ]:
#@title 🎧 Before You Start: Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_todo_1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 1: Build a sync checker
#
# Steps:
# 1. Implement check_sync(models) that returns True if all models have
#    identical parameters, False otherwise
# 2. Deliberately break sync by skipping AllReduce for one GPU
# 3. Show that the checker detects the desynchronization
# 4. Fix it by re-broadcasting weights from GPU 0

def check_sync(models):
    """Check if all model replicas have identical parameters."""
    # TODO: Compare parameters of models[0] with all others
    # Return (is_synced: bool, max_diff: float)
    pass

def broadcast_from_rank0(models):
    """Fix desync by copying weights from models[0] to all others."""
    # TODO: Copy state_dict from models[0] to models[1], models[2], ...
    pass

# Test:
# 1. Create 4 identical models
# 2. Run one training step WITH AllReduce -> check_sync should return True
# 3. Run one training step WITHOUT AllReduce on GPU 2 -> check_sync should return False
# 4. Fix with broadcast_from_rank0 -> check_sync should return True again

In [ ]:
#@title 🎧 Narration: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_todo_intro_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### TODO Exercise 2: Communication-Computation Overlap

In real DDP, the AllReduce can be overlapped with the backward pass (gradient bucketing). Simulate this by measuring how much time could be saved.

In [ ]:
#@title 🎧 Before You Start: Todo
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_todo_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO Exercise 2: Simulate communication-computation overlap
#
# Steps:
# 1. For a model with parameters p1, p2, ..., pN:
#    - Group them into B buckets
#    - Compute backward for each bucket
#    - Measure time for backward and AllReduce separately
# 2. Calculate:
#    a. Sequential time: total_backward + total_allreduce
#    b. Overlapped time: max(total_backward, total_allreduce)
#    c. Savings = (sequential - overlapped) / sequential * 100
#
# This is a thought exercise -- estimate the times from:
#   - Backward: proportional to FLOPs
#   - AllReduce: proportional to parameter bytes / bandwidth

model_sizes_B = [0.1, 0.5, 1.0, 5.0, 10.0]  # in billions
bandwidth_GBps = 25  # GB/s NVLink
compute_rate_TFLOPS = 100  # T4 ~ 65 TFLOPS, A100 ~ 312 TFLOPS

print(f"{'Model Size':>12} {'Backward (ms)':>14} {'AllReduce (ms)':>15} {'Sequential':>12} {'Overlapped':>12} {'Savings':>8}")
print("-" * 80)

for size in model_sizes_B:
    # Rough estimates:
    # Backward FLOPs ~ 4 * params (2 for forward, 2 for backward with activations)
    flops = 4 * size * 1e9
    backward_ms = flops / (compute_rate_TFLOPS * 1e12) * 1000

    # AllReduce bytes ~ 2 * params * 2 (FP16, ring algorithm factor)
    allreduce_bytes = 2 * size * 1e9 * 2
    allreduce_ms = allreduce_bytes / (bandwidth_GBps * 1e9) * 1000

    sequential = backward_ms + allreduce_ms
    overlapped = max(backward_ms, allreduce_ms)
    savings = (sequential - overlapped) / sequential * 100

    print(f"{size:>10.1f}B {backward_ms:>14.1f} {allreduce_ms:>15.1f} {sequential:>12.1f} {overlapped:>12.1f} {savings:>7.1f}%")

# TODO: Which model sizes benefit most from overlap?
# TODO: What happens when bandwidth increases (e.g., NVLink vs PCIe)?

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## Summary

In this notebook, we have:

1. **Implemented data parallelism from scratch** -- replicate, shard, compute, AllReduce, update

2. **Verified mathematical equivalence** -- data parallel training produces identical results to single-GPU training

3. **Analyzed communication overhead** -- AllReduce cost grows with model size, limiting scaling efficiency

4. **Built a complete training loop** -- simulated 4-GPU data parallel training on a single GPU

5. **Combined all three techniques** -- checkpointing + accumulation + data parallelism for the complete memory toolkit

### Key Takeaways

- Data parallelism replicates the full model on every GPU -- simple but memory-intensive
- AllReduce averages gradients so all replicas stay synchronized
- Effective batch size = `num_gpus * micro_batch_per_gpu`
- Scaling efficiency decreases with more GPUs (communication overhead)
- The three techniques compose orthogonally:
  - **Checkpointing**: reduces memory per layer
  - **Accumulation**: simulates large batches
  - **Data parallelism**: multiplies throughput

### What Comes Next

Data parallelism has a fundamental limitation: **every GPU holds a complete model copy**. For models too large to fit on a single GPU (70B+ parameters), we need to split the model itself across GPUs. This leads us to **model parallelism** -- tensor parallelism, pipeline parallelism, and ZeRO optimizer -- the topics of the next pods in this course.